## Создаем Spark-сессию

In [13]:
import os
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.window import Window


spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

In [17]:
spark

## Подключаемся к источникам

### csv

In [14]:
path = "s3a://dev/volchara/data"


In [15]:
campaigns_dict = (
    spark.read
    .option("header", True)
    .csv(f"{path}/campaigns_dict.csv")
)

In [16]:
# ленивые вычисления (transformations, actions)
campaigns_dict.show(5, truncate=False)

+-----------+------------------------------------------+
|campaign_id|campaign_name                             |
+-----------+------------------------------------------+
|1          |year_modern_kitchen_launch_20250115       |
|2          |quarter_custom_kitchens_showcase_20240210 |
|3          |month_smart_kitchen_promotion_20240305    |
|4          |year_luxury_kitchens_exhibit_20240420     |
|5          |quarter_ecofriendly_kitchen_offer_20240512|
+-----------+------------------------------------------+
only showing top 5 rows



In [18]:
campaigns_dict.printSchema()

root
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)



### parquet

In [19]:
submits = spark.read.parquet(f'{path}/submits.parquet')

In [20]:
submits.show(5, truncate=False)

+---------+--------+-----------+
|submit_id|name    |phone      |
+---------+--------+-----------+
|2282     |Jennifer|79511904041|
|9898     |Jeffrey |79824419733|
|9005     |Linda   |79074725672|
|1507     |Teresa  |79864203598|
|3803     |Tanya   |79779567654|
+---------+--------+-----------+
only showing top 5 rows



In [21]:
submits.printSchema()

root
 |-- submit_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- phone: long (nullable = true)



In [23]:
deals = spark.read.parquet(f'{path}/deals.parquet')

In [24]:
deals.show(5, truncate=False)

+-------+----------+---------------------+-----------+------------------------+-----------------------------------------------------+
|deal_id|deal_date |fio                  |phone      |email                   |address                                              |
+-------+----------+---------------------+-----------+------------------------+-----------------------------------------------------+
|1      |2024-03-04|Gregory Wu           |79746561889|paul80@example.net      |098 Yates Cliff Apt. 241, East Monica, DE 88076      |
|2      |2024-08-20|William Ross Jr.     |79074725672|xyoung@example.org      |197 Willie Groves Apt. 655, Port Angelaberg, LA 39384|
|3      |2024-10-15|Sonya Kerr           |79201244835|elewis@example.com      |144 Andrew Cape, Lake Nicholas, SC 58918             |
|4      |2024-12-31|Mrs. Angela Tucker MD|79771829751|robertparker@example.net|6056 Collins View, South Harold, OR 15650            |
|5      |2024-03-23|Eric Flores          |79729054809|barbara7

In [25]:
deals.show(2, truncate=False, vertical=True)

-RECORD 0----------------------------------------------------------
 deal_id   | 1                                                     
 deal_date | 2024-03-04                                            
 fio       | Gregory Wu                                            
 phone     | 79746561889                                           
 email     | paul80@example.net                                    
 address   | 098 Yates Cliff Apt. 241, East Monica, DE 88076       
-RECORD 1----------------------------------------------------------
 deal_id   | 2                                                     
 deal_date | 2024-08-20                                            
 fio       | William Ross Jr.                                      
 phone     | 79074725672                                           
 email     | xyoung@example.org                                    
 address   | 197 Willie Groves Apt. 655, Port Angelaberg, LA 39384 
only showing top 2 rows



In [26]:
deals.printSchema()

root
 |-- deal_id: long (nullable = true)
 |-- deal_date: string (nullable = true)
 |-- fio: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- email: string (nullable = true)
 |-- address: string (nullable = true)



### postgres

In [46]:
pg_host = 'postgres_source'
pg_port = '5432'
pg_db = 'source'
pg_table = 'webinar.costs'
pg_user = os.getenv("POSTGRES_USER")
pg_password = os.getenv("POSTGRES_PASSWORD")

costs = (
    spark.read
    .format('jdbc')
    .option('url', f'jdbc:postgresql://{pg_host}:{pg_port}/{pg_db}')
    .option('dbtable', pg_table)
    .option('user', pg_user)
    .option('password', pg_password)
    .option('fetchsize', 1000)
    .option('driver', 'org.postgresql.Driver')
    .load()
)


jdbc_df.show()

+----------+-----------+------+------+-----+
|      date|campaign_id| costs|clicks|views|
+----------+-----------+------+------+-----+
|2024-01-01|          1|670.52|    40|  110|
|2024-01-01|          2| 602.5|    11|  849|
|2024-01-01|          3|654.74|    51|  566|
|2024-01-01|          4|897.24|    86|  679|
|2024-01-01|          5|758.19|    30|  585|
|2024-01-01|          6|523.91|    77|  883|
|2024-01-01|          7|465.35|    98|  527|
|2024-01-01|          8|771.47|     4|  585|
|2024-01-01|          9|973.09|    51|  255|
|2024-01-01|         10|886.07|    88|  815|
|2024-01-01|         11|489.74|    54|  624|
|2024-01-01|         12|522.04|    25|  898|
|2024-01-01|         13|254.72|    23|  895|
|2024-01-01|         14| 840.0|    89|  302|
|2024-01-01|         15|420.02|    64|  974|
|2024-01-01|         16|783.93|    38|  202|
|2024-01-01|         17| 86.72|    30|  554|
|2024-01-01|         18|480.09|    41|  484|
|2024-01-01|         19|856.34|    95|  268|
|2024-01-0

In [53]:
costs = (
    spark.read
    .format('jdbc')
    .option('url', f'jdbc:postgresql://{pg_host}:{pg_port}/{pg_db}')
    .option('dbtable', pg_table)
    .option('user', pg_user)
    .option('password', pg_password)
    .option('driver', 'org.postgresql.Driver')
    .load()
)

In [54]:
costs.show(5, truncate=False)  # заглянуть в pgAdmin

+----------+-----------+------+------+-----+
|date      |campaign_id|costs |clicks|views|
+----------+-----------+------+------+-----+
|2024-01-01|1          |670.52|40    |110  |
|2024-01-01|2          |602.5 |11    |849  |
|2024-01-01|3          |654.74|51    |566  |
|2024-01-01|4          |897.24|86    |679  |
|2024-01-01|5          |758.19|30    |585  |
+----------+-----------+------+------+-----+
only showing top 5 rows



In [55]:
costs.printSchema()

root
 |-- date: date (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- costs: float (nullable = true)
 |-- clicks: integer (nullable = true)
 |-- views: integer (nullable = true)



### clickhouse

In [59]:
ch_host = 'clickhouse01'
ch_port = '8123'
ch_db = 'volchara'
ch_table = 'visits'


In [60]:
visits = (
    spark.read
    .format('jdbc')
    .option('url', f'jdbc:clickhouse://{ch_host}:{ch_port}/{ch_db}')
    .option('dbtable', ch_table)
    .option('driver', 'com.clickhouse.jdbc.ClickHouseDriver')
    .load()
)

In [61]:
visits.show(5)

+-------+-------------------+--------------------+--------+--------+--------+--------------------+----------------+
|visitid|      visitDateTime|                 URL|duration|clientID|  source|         UTMCampaign|          params|
+-------+-------------------+--------------------+--------+--------+--------+--------------------+----------------+
| 100059|2024-08-31 14:08:00|https://our-cool-...|      79|     522|  direct|month_openconcept...|              []|
| 100094|2024-06-09 09:30:12|https://our-cool-...|      20|     847|internal|month_contemporar...|              []|
| 100109|2024-04-03 17:07:17|https://our-cool-...|      66|     121|  direct|year_traditional_...|['submit', 4315]|
| 100150|2024-02-02 18:21:08|https://our-cool-...|      12|     790|  direct|month_contemporar...|              []|
| 100164|2024-07-13 15:56:27|https://our-cool-...|      44|     958|  direct|year_luxury_kitch...|['submit', 7580]|
+-------+-------------------+--------------------+--------+--------+----

In [62]:
visits.show(1, truncate=False, vertical=True)

-RECORD 0------------------------------------------------------
 visitid       | 100059                                        
 visitDateTime | 2024-08-31 14:08:00                           
 URL           | https://our-cool-website.com/featured         
 duration      | 79                                            
 clientID      | 522                                           
 source        | direct                                        
 UTMCampaign   | month_openconcept_kitchens_promotion_20240628 
 params        | []                                            
only showing top 1 row



In [63]:
visits.printSchema()

root
 |-- visitid: integer (nullable = true)
 |-- visitDateTime: timestamp (nullable = true)
 |-- URL: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- clientID: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- UTMCampaign: string (nullable = true)
 |-- params: string (nullable = true)



In [ ]:
# методология и моника

## Готовим источники

### Визиты (clickhouse)

In [ ]:
visits.show(1)

In [64]:
# кликхаус
filtered_step1 = (
    visits
    .withColumn('dt', F.date_format(F.col('visitDateTime'), 'yyyy-MM-dd'))
    .where(F.col('dt').between('2024-01-01', '2025-01-27'))
    .where(F.col('source').isin('ad', 'direct'))
    .where(F.col('URL').rlike('.*checkout.*|.*add.*|.*home.*|.*contact.*|.*top50.*|.*customer-service.*|.*wishlist.*|.*sale.*|.*best-sellers.*|.*view.*|.*discount.*|.*featured.*|.*new-arrivals.*|.*settings.*|.*return-policy.*|.*edit.*|.*delete.*|.*reviews.*|.*products.*|.*about.*'))
    .select(
        'dt',
        'visitid',
        'clientID',
        'URL',
        'duration',
        'source',
        'UTMCampaign',
        'params',
        F.regexp_replace(F.col('params'), r'\[|\]', '').alias('params_regex')
    )
    .withColumn('params_split', F.split('params_regex', ', '))
)

In [65]:
filtered_step1.show(1, truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------
 dt           | 2024-08-31                                    
 visitid      | 100059                                        
 clientID     | 522                                           
 URL          | https://our-cool-website.com/featured         
 duration     | 79                                            
 source       | direct                                        
 UTMCampaign  | month_openconcept_kitchens_promotion_20240628 
 params       | []                                            
 params_regex |                                               
 params_split | []                                            
only showing top 1 row



In [66]:
filtered_step1.printSchema()  # string vs array

root
 |-- dt: string (nullable = true)
 |-- visitid: integer (nullable = true)
 |-- clientID: integer (nullable = true)
 |-- URL: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- UTMCampaign: string (nullable = true)
 |-- params: string (nullable = true)
 |-- params_regex: string (nullable = true)
 |-- params_split: array (nullable = true)
 |    |-- element: string (containsNull = false)



In [67]:
filtered_step2 = (
    filtered_step1
    .withColumn('event_type', F.regexp_replace(F.col('params_split')[0], "'", ''))
    .withColumn('event_id', F.col('params_split')[1].cast('int'))
)

In [69]:
filtered_step2.show(1, truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------
 dt           | 2024-08-31                                    
 visitid      | 100059                                        
 clientID     | 522                                           
 URL          | https://our-cool-website.com/featured         
 duration     | 79                                            
 source       | direct                                        
 UTMCampaign  | month_openconcept_kitchens_promotion_20240628 
 params       | []                                            
 params_regex |                                               
 params_split | []                                            
 event_type   |                                               
 event_id     | NULL                                          
only showing top 1 row



In [70]:
filtered_step2.printSchema()

root
 |-- dt: string (nullable = true)
 |-- visitid: integer (nullable = true)
 |-- clientID: integer (nullable = true)
 |-- URL: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- UTMCampaign: string (nullable = true)
 |-- params: string (nullable = true)
 |-- params_regex: string (nullable = true)
 |-- params_split: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- event_type: string (nullable = true)
 |-- event_id: integer (nullable = true)



In [71]:
visits_df = (
    filtered_step2
    .where(F.col('event_type') == 'submit')
    .select(
        'dt',
        F.col('visitid').cast('string').alias('visitid'),
        F.col('clientID').cast('string').alias('clientid'),
        'URL',
        'duration',
        'source',
        'UTMCampaign',
        'event_type',
        'event_id'
    )
    .distinct()
)

In [72]:
visits_df.show(1, truncate=False, vertical=True)

-RECORD 0----------------------------------------------
 dt          | 2024-04-03                              
 visitid     | 100109                                  
 clientid    | 121                                     
 URL         | https://our-cool-website.com/view       
 duration    | 66                                      
 source      | direct                                  
 UTMCampaign | year_traditional_kitchens_fair_20240415 
 event_type  | submit                                  
 event_id    | 4315                                    
only showing top 1 row



In [130]:
visits_df.printSchema()

root
 |-- dt: string (nullable = true)
 |-- visitid: string (nullable = true)
 |-- clientid: string (nullable = true)
 |-- URL: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- UTMCampaign: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_id: integer (nullable = true)



In [131]:
visits_df.count()

2433

In [132]:
visits.count()

10000

### Расходы (postgres)

In [73]:
costs_df = (
    costs
    .groupBy(
        F.col('date').cast('string').alias('date'),
        'campaign_id'
    )
    .agg(
        F.sum(F.col('costs')).cast('decimal(19,2)').alias('costs'),
        F.sum(F.col('clicks')).alias('clicks'),
        F.sum(F.col('views')).alias('views')
    )
)

In [74]:
costs_df.show(5)

+----------+-----------+------+------+-----+
|      date|campaign_id| costs|clicks|views|
+----------+-----------+------+------+-----+
|2024-01-05|         75|107.03|    26|  958|
|2024-01-05|         97|408.51|    59|  292|
|2024-01-06|         44|748.68|    33|  380|
|2024-01-06|         58|422.91|    30|  575|
|2024-01-11|         91|496.45|     5|  435|
+----------+-----------+------+------+-----+
only showing top 5 rows



In [75]:
costs_df.printSchema()

root
 |-- date: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- costs: decimal(19,2) (nullable = true)
 |-- clicks: long (nullable = true)
 |-- views: long (nullable = true)



### Кампании (csv)

In [77]:
campaigns_dict.show(1, truncate=False)

+-----------+-----------------------------------+
|campaign_id|campaign_name                      |
+-----------+-----------------------------------+
|1          |year_modern_kitchen_launch_20250115|
+-----------+-----------------------------------+
only showing top 1 row



In [78]:
campaigns_dict.printSchema()

root
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)



In [79]:
campaigns_df = (
    campaigns_dict
    .withColumn('campaign_id', F.col('campaign_id').cast('integer'))
    .withColumn(
        'campaign_duration',
        F.when(F.col('campaign_name').like('year%'), 'Год')
        .when(F.col('campaign_name').like('quarter%'), 'Квартал')
        .when(F.col('campaign_name').like('month%'), 'Месяц')
        .otherwise(None)
    )
)

In [80]:
campaigns_df.show(5, truncate=False)

+-----------+------------------------------------------+-----------------+
|campaign_id|campaign_name                             |campaign_duration|
+-----------+------------------------------------------+-----------------+
|1          |year_modern_kitchen_launch_20250115       |Год              |
|2          |quarter_custom_kitchens_showcase_20240210 |Квартал          |
|3          |month_smart_kitchen_promotion_20240305    |Месяц            |
|4          |year_luxury_kitchens_exhibit_20240420     |Год              |
|5          |quarter_ecofriendly_kitchen_offer_20240512|Квартал          |
+-----------+------------------------------------------+-----------------+
only showing top 5 rows



In [81]:
campaigns_df.printSchema()

root
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_duration: string (nullable = true)



### Заявки (parquet)

In [82]:
submits.printSchema()

root
 |-- submit_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- phone: long (nullable = true)



In [ ]:
submits.show(1)

In [83]:
submits_df = (
    submits
    .withColumn('phone', F.col('phone').cast('string'))
    .withColumn('phone_plus', F.concat(F.lit('+'), F.col('phone')))
    .withColumn('phone_md5', F.md5('phone'))
    .withColumn('phone_plus_md5', F.md5('phone_plus'))
)

In [84]:
submits_df.show(2, truncate=False)

+---------+--------+-----------+------------+--------------------------------+--------------------------------+
|submit_id|name    |phone      |phone_plus  |phone_md5                       |phone_plus_md5                  |
+---------+--------+-----------+------------+--------------------------------+--------------------------------+
|2282     |Jennifer|79511904041|+79511904041|4c7720fdf6f9eec623dc0f961f31f488|6ce81d5347bcd3eadb2921b7c4828e3b|
|9898     |Jeffrey |79824419733|+79824419733|9d106e45036bd4176774eb94adc9aacc|43630233dcc965f9827e394038b0321a|
+---------+--------+-----------+------------+--------------------------------+--------------------------------+
only showing top 2 rows



### Сделки (parquet)

In [85]:
deals.printSchema()

root
 |-- deal_id: long (nullable = true)
 |-- deal_date: string (nullable = true)
 |-- fio: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- email: string (nullable = true)
 |-- address: string (nullable = true)



In [86]:
deals_df = (
    deals
    .withColumn('username', F.split(F.col('email'), '@').getItem(0))
    .withColumn('domain', F.split(F.col('email'), '@').getItem(1))
    .where(F.col('domain').isin('example.com', 'example.org', 'example.net'))
    .withColumn('phone', F.col('phone').cast('string'))
)

In [87]:
deals_df.show(1)

+-------+----------+----------+-----------+------------------+--------------------+--------+-----------+
|deal_id| deal_date|       fio|      phone|             email|             address|username|     domain|
+-------+----------+----------+-----------+------------------+--------------------+--------+-----------+
|      1|2024-03-04|Gregory Wu|79746561889|paul80@example.net|098 Yates Cliff A...|  paul80|example.net|
+-------+----------+----------+-----------+------------------+--------------------+--------+-----------+
only showing top 1 row



In [88]:
deals_df.printSchema()

root
 |-- deal_id: long (nullable = true)
 |-- deal_date: string (nullable = true)
 |-- fio: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- address: string (nullable = true)
 |-- username: string (nullable = true)
 |-- domain: string (nullable = true)



## Собираем витрину

In [89]:
# концепция "One Big Table"
customer_detailed = (
    visits_df.alias('v')
    .join(
        submits_df.alias('s'),
        F.col('v.event_id') == F.col('s.submit_id'),
        'left'
    )
    .join(
        deals_df.alias('d'),
        (F.col('s.phone') == F.col('d.phone')) &
        (F.col('v.dt') <= F.col('d.deal_date')),
        'left'
    )
    .join(
        campaigns_df.alias('camp'),
        F.col('v.utmcampaign') == F.col('camp.campaign_name'),
        'left'
    )
    .join(
        costs_df.alias('c'),
        (F.col('camp.campaign_id') == F.col('c.campaign_id')) &
        (F.col('v.dt') == F.col('c.date')),
        'left'
    )
    .select(
        'v.dt',
        F.col('v.visitid').alias('visit_id'),
        F.col('v.clientid').alias('client_id'),
        'v.url',
        'v.duration',
        'v.source',
        'v.utmcampaign',
        'v.event_type',
        'v.event_id',
        's.submit_id',
        's.name',
        's.phone',
        's.phone_plus',
        's.phone_md5',
        's.phone_plus_md5',
        'd.deal_id',
        'd.deal_date',
        'd.fio',
        F.col('d.phone').alias('phone_deal'),
        'd.email',
        'd.address',
        'd.username',
        'd.domain',
        'camp.campaign_name',
        'camp.campaign_duration',
        'c.costs',
        'c.clicks',
        'c.views'
    )
)

In [90]:
customer_detailed.printSchema()

root
 |-- dt: string (nullable = true)
 |-- visit_id: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- utmcampaign: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_id: integer (nullable = true)
 |-- submit_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- phone_plus: string (nullable = true)
 |-- phone_md5: string (nullable = true)
 |-- phone_plus_md5: string (nullable = true)
 |-- deal_id: long (nullable = true)
 |-- deal_date: string (nullable = true)
 |-- fio: string (nullable = true)
 |-- phone_deal: string (nullable = true)
 |-- email: string (nullable = true)
 |-- address: string (nullable = true)
 |-- username: string (nullable = true)
 |-- domain: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_duration: string (nullable = tr

In [91]:
len(customer_detailed.columns)

28

In [92]:
customer_detailed.cache()

26/04/27 18:11:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[dt: string, visit_id: string, client_id: string, url: string, duration: int, source: string, utmcampaign: string, event_type: string, event_id: int, submit_id: bigint, name: string, phone: string, phone_plus: string, phone_md5: string, phone_plus_md5: string, deal_id: bigint, deal_date: string, fio: string, phone_deal: string, email: string, address: string, username: string, domain: string, campaign_name: string, campaign_duration: string, costs: decimal(19,2), clicks: bigint, views: bigint]

In [93]:
customer_detailed.count()  # Spark UI

2639

In [94]:
campaigns_agg = (
    customer_detailed
    .groupBy('campaign_name')
    .agg(
        F.countDistinct('visit_id').alias('unique_visits'),
        F.countDistinct('client_id').alias('unique_clients'),
        F.countDistinct('submit_id').alias('unique_submits'),
        F.countDistinct('deal_id').alias('unique_deals'),
        F.sum('costs').alias('total_costs'),
        F.sum('clicks').alias('total_clicks'),
        F.sum('views').alias('total_views'),
        F.sum('duration').alias('total_duration')
    )
    .withColumn('avg_deal_cost', (F.col('total_costs') / F.col('unique_deals')).cast('decimal(19,2)'))
)

In [95]:
campaigns_agg.cache().count()

99

In [96]:
campaigns_agg.show(1, truncate=False, vertical=True)

-RECORD 0-------------------------------------------------
 campaign_name  | year_traditional_kitchens_demo_20240725 
 unique_visits  | 22                                      
 unique_clients | 21                                      
 unique_submits | 5                                       
 unique_deals   | 1                                       
 total_costs    | 11446.75                                
 total_clicks   | 1441                                    
 total_views    | 14004                                   
 total_duration | 1100                                    
 avg_deal_cost  | 11446.75                                
only showing top 1 row



In [97]:
dates_agg = (
    customer_detailed
    .groupBy(F.substring('dt', 1, 7).alias('month'))  # 2025-01-01
    .agg(
        F.countDistinct('visit_id').alias('unique_visits'),
        F.countDistinct('client_id').alias('unique_clients'),
        F.countDistinct('submit_id').alias('unique_submits'),
        F.countDistinct('deal_id').alias('unique_deals'),
        F.sum('costs').alias('total_costs'),
        F.sum('clicks').alias('total_clicks'),
        F.sum('views').alias('total_views'),
        F.sum('duration').alias('total_duration')
    )
    .withColumn('avg_deal_cost', (F.col('total_costs') / F.col('unique_deals')).cast('decimal(19,2)'))
)

In [98]:
dates_agg.cache().count()

13

In [99]:
dates_agg.show(1, truncate=False, vertical=True)

-RECORD 0-------------------
 month          | 2024-09   
 unique_visits  | 191       
 unique_clients | 168       
 unique_submits | 67        
 unique_deals   | 15        
 total_costs    | 100807.61 
 total_clicks   | 10091     
 total_views    | 102230    
 total_duration | 10225     
 avg_deal_cost  | 6720.51   
only showing top 1 row



In [103]:
def save_to_postgres(df, table_name):
    (
        df.write
        .format('jdbc')
        .option('url', f'jdbc:postgresql://{pg_host}:{pg_port}/{pg_db}')
        .option('dbtable', table_name)
        .option('user', pg_user)
        .option('password', pg_password)
        .option('driver', 'org.postgresql.Driver')
        .mode("overwrite")
        .save()
    )

In [107]:
save_to_postgres(customer_detailed, 'webinar.customer_detailed')

In [108]:
save_to_postgres(campaigns_agg, 'webinar.campaigns_agg')

In [109]:
save_to_postgres(dates_agg, 'webinar.dates_agg')

## Считаем метрики и анализируем результаты

In [110]:
# 1. Кампании без выручки
(
    campaigns_agg
    .where('unique_deals = 0')
    .select('campaign_name', 'total_costs', 'unique_deals')
    .sort('total_costs')
    .show(truncate=False)
)

+-----------------------------------------------+-----------+------------+
|campaign_name                                  |total_costs|unique_deals|
+-----------------------------------------------+-----------+------------+
|month_smart_kitchens_launch_20240330           |7417.29    |0           |
|quarter_custom_kitchens_showcase_20240210      |8397.63    |0           |
|month_contemporary_kitchens_event_20241208     |9077.02    |0           |
|month_contemporary_kitchens_event_20240920     |10256.14   |0           |
|month_ecofriendly_kitchens_experience_20241219 |10682.85   |0           |
|quarter_ecofriendly_kitchen_experience_20241119|10999.04   |0           |
|month_smart_kitchen_promotion_20240305         |11430.88   |0           |
|year_traditional_kitchens_showcase_20241018    |12105.87   |0           |
|year_modern_kitchens_demo_20240419             |12427.38   |0           |
|month_openconcept_kitchens_initiative_20240328 |13187.82   |0           |
|quarter_luxury_kitchens_

In [111]:
(
    campaigns_agg
    .where('unique_deals = 0')
    .select(F.count('campaign_name'), F.sum('total_costs'))
    .show()
)

+--------------------+----------------+
|count(campaign_name)|sum(total_costs)|
+--------------------+----------------+
|                  13|       149956.85|
+--------------------+----------------+



In [112]:
# 2. Средняя цена сделки
(
    campaigns_agg
    .where('unique_deals > 0')
    .select('campaign_name', 'total_costs', 'unique_deals', 'avg_deal_cost')
    .sort('avg_deal_cost')
    .show(truncate=False)
)

+------------------------------------------------+-----------+------------+-------------+
|campaign_name                                   |total_costs|unique_deals|avg_deal_cost|
+------------------------------------------------+-----------+------------+-------------+
|year_modern_kitchens_showcase_20241014          |11705.53   |8           |1463.19      |
|quarter_custom_kitchens_innovation_20240817     |15356.76   |10          |1535.68      |
|year_modern_kitchen_experience_20240702         |8528.47    |5           |1705.69      |
|quarter_custom_kitchens_experience_20240222     |16131.74   |9           |1792.42      |
|month_contemporary_kitchens_promotion_20241215  |14908.49   |8           |1863.56      |
|quarter_custom_kitchen_initiative_20241102      |16944.70   |8           |2118.09      |
|month_openconcept_kitchens_promotion_20240628   |13101.37   |6           |2183.56      |
|year_smart_kitchens_event_20250102              |11699.64   |5           |2339.93      |
|year_trad

In [113]:
(
    campaigns_agg
    .where('unique_deals > 0')
    .select('campaign_name', 'total_costs', 'unique_deals', 'avg_deal_cost')
    .sort('avg_deal_cost', ascending=False)
    .show(truncate=False)
)

+-----------------------------------------------+-----------+------------+-------------+
|campaign_name                                  |total_costs|unique_deals|avg_deal_cost|
+-----------------------------------------------+-----------+------------+-------------+
|quarter_spacesaving_kitchen_innovation_20240825|16861.94   |1           |16861.94     |
|month_contemporary_kitchen_showcase_20240630   |15880.89   |1           |15880.89     |
|quarter_custom_kitchen_show_20240213           |15874.75   |1           |15874.75     |
|year_luxury_kitchens_show_20241001             |14752.07   |1           |14752.07     |
|quarter_spacesaving_kitchen_showcase_20240228  |14382.19   |1           |14382.19     |
|month_custom_kitchens_show_20241205            |13947.35   |1           |13947.35     |
|year_traditional_kitchens_launch_20240707      |13857.96   |1           |13857.96     |
|year_traditional_kitchens_experience_20240715  |12925.29   |1           |12925.29     |
|quarter_spacesaving_

In [114]:
# 3. Убыточные кампании. Пусть каждая сделка стоит 5к
(
    campaigns_agg
    .select('campaign_name', 'total_costs', 'unique_deals', 'avg_deal_cost')
    .withColumn('revenue', F.col('unique_deals') * F.lit(5000))
    .withColumn('profit', F.col('revenue') - F.col('total_costs'))
    .sort('profit')
    .show(truncate=False)
)

+-----------------------------------------------+-----------+------------+-------------+-------+---------+
|campaign_name                                  |total_costs|unique_deals|avg_deal_cost|revenue|profit   |
+-----------------------------------------------+-----------+------------+-------------+-------+---------+
|quarter_custom_kitchens_experience_20240527    |15732.40   |0           |NULL         |0      |-15732.40|
|month_openconcept_kitchens_experience_20240910 |14837.06   |0           |NULL         |0      |-14837.06|
|quarter_luxury_kitchens_innovation_20241116    |13405.47   |0           |NULL         |0      |-13405.47|
|month_openconcept_kitchens_initiative_20240328 |13187.82   |0           |NULL         |0      |-13187.82|
|year_modern_kitchens_demo_20240419             |12427.38   |0           |NULL         |0      |-12427.38|
|year_traditional_kitchens_showcase_20241018    |12105.87   |0           |NULL         |0      |-12105.87|
|quarter_spacesaving_kitchen_innovati

In [115]:
# 4. Самые прибыльные кампании
(
    campaigns_agg
    .select('campaign_name', 'total_costs', 'unique_deals', 'avg_deal_cost')
    .withColumn('revenue', F.col('unique_deals') * F.lit(5000))
    .withColumn('profit', F.col('revenue') - F.col('total_costs'))
    .sort(F.desc('profit'))
    .show(truncate=False)
)

+------------------------------------------------+-----------+------------+-------------+-------+--------+
|campaign_name                                   |total_costs|unique_deals|avg_deal_cost|revenue|profit  |
+------------------------------------------------+-----------+------------+-------------+-------+--------+
|quarter_custom_kitchens_innovation_20240817     |15356.76   |10          |1535.68      |50000  |34643.24|
|quarter_custom_kitchens_experience_20240222     |16131.74   |9           |1792.42      |45000  |28868.26|
|year_modern_kitchens_showcase_20241014          |11705.53   |8           |1463.19      |40000  |28294.47|
|month_contemporary_kitchens_promotion_20241215  |14908.49   |8           |1863.56      |40000  |25091.51|
|quarter_custom_kitchen_initiative_20241102      |16944.70   |8           |2118.09      |40000  |23055.30|
|year_traditional_kitchens_fair_20240415         |19396.53   |8           |2424.57      |40000  |20603.47|
|month_openconcept_kitchens_promotion

In [116]:
# 5. Метрики в разбивке по месяцам
(
    dates_agg
    .withColumn('revenue', F.col('unique_deals') * F.lit(5000))
    .withColumn('profit', F.col('revenue') - F.col('total_costs'))
    .drop('total_duration', 'total_views')
    .sort('month')
    .show()
)

+-------+-------------+--------------+--------------+------------+-----------+------------+-------------+-------+---------+
|  month|unique_visits|unique_clients|unique_submits|unique_deals|total_costs|total_clicks|avg_deal_cost|revenue|   profit|
+-------+-------------+--------------+--------------+------------+-----------+------------+-------------+-------+---------+
|2024-01|          204|           185|            68|          36|  114500.26|       11503|      3180.56| 180000| 65499.74|
|2024-02|          184|           164|            55|          28|   98550.71|        8960|      3519.67| 140000| 41449.29|
|2024-03|          182|           164|            57|          34|   99016.38|       10123|      2912.25| 170000| 70983.62|
|2024-04|          187|           166|            53|          21|   99450.75|        9704|      4735.75| 105000|  5549.25|
|2024-05|          188|           174|            68|          30|   98370.96|       10277|      3279.03| 150000| 51629.04|
|2024-06

In [117]:
# 6. Сколько всего потратили денег на рекламу за год
dates_agg.select(F.sum('total_costs')).show()

+----------------+
|sum(total_costs)|
+----------------+
|      1319918.78|
+----------------+



## Освобождаем ресурсы, останавливаем Spark-сессию

In [180]:
# Spark UI
customer_detailed.unpersist()
campaigns_agg.unpersist()
dates_agg.unpersist()

DataFrame[month: string, unique_visits: bigint, unique_clients: bigint, unique_submits: bigint, unique_deals: bigint, total_costs: decimal(29,2), total_clicks: bigint, total_views: bigint, total_duration: bigint, avg_deal_cost: decimal(19,2)]

In [181]:
spark.stop()